In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotnine as p9
import scipy.stats
from PIL import Image
from tqdm.notebook import tqdm

In [ ]:
# prepare data
noisy_labels_dir = Path('../data/external/dataset/noisy_labels')
fp_noisy_labels = sorted(list(noisy_labels_dir.rglob('*.png')))
clean_labels_dir = Path('../data/external/dataset/clean_labels')
fp_clean_labels = sorted(list(clean_labels_dir.rglob('*.png')))
imgs_dir = Path('../data/external/dataset/image_patches/')
fp_imgs = sorted(list(imgs_dir.rglob('*.png')))
assert len(fp_noisy_labels) == len(fp_clean_labels) == len(fp_imgs) == 5000

# map the filenames to the filepaths
fp_imgs = {fp.stem: fp for fp in fp_imgs}
fp_noisy_labels = {fp.stem: fp for fp in fp_noisy_labels}
fp_clean_labels = {fp.stem: fp for fp in fp_clean_labels}

In [ ]:
def compute_image_level_metrics(img_pred, img_true):
    """
    Compute IoU, Precision, Recall, Accuracy and F1 score between predicted mask and ground truth mask.
    """

    # make sure pred and mask are boolean arrays
    assert img_pred.dtype == bool, "Predicted mask should be a boolean array."
    assert img_true.dtype == bool, "Ground truth mask should be a boolean array."

    tp = np.sum(img_true & img_pred)
    tn = np.sum(~img_true & ~img_pred)
    fp = np.sum(~img_true & img_pred)
    fn = np.sum(img_true & ~img_pred)

    iou = tp / max(tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    f1_score = 2 * tp / max(2 * tp + fp + fn, 1)

    stats = {
        'iou': iou,
        'precision': precision,
        'recall': recall,
        'accuracy': accuracy,
        'f1_score': f1_score
    }

    return stats

### Compute the ground truth ranking (i.e. based on the IOU between noisy labels and the clean ones)

In [ ]:
# TODO: check if this is the same as the one used for the competition as we're seeing some differences in the final scores
stats_noisy_vs_clean_all = []
for img_name in tqdm(fp_clean_labels):
    mask_noisy = np.array(Image.open(fp_noisy_labels[img_name])) > 0
    mask_clean = np.array(Image.open(fp_clean_labels[img_name])) > 0
    stats_noisy_vs_clean = compute_image_level_metrics(mask_clean, mask_noisy)
    stats_noisy_vs_clean['img'] = img_name
    stats_noisy_vs_clean_all.append(stats_noisy_vs_clean)
df_noisy_vs_clean = pd.DataFrame(stats_noisy_vs_clean_all)
df_noisy_vs_clean

In [ ]:
def compute_kendall_tau(sdf):
    if 'img' not in sdf.columns:
        raise RuntimeError('sdf must contain "img" column.')
    if 'iou' not in sdf.columns:
        raise RuntimeError('sdf must contain "iou" column.')

    if not len(sdf) == len(fp_clean_labels):
        display(sdf)
        raise RuntimeError(
            f"The input dataframe should contain stats for all images. "
            f"Got {len(sdf)} instead of {len(fp_clean_labels)}."
        )
    # make sure the two dataframes are aligned
    if not all(sdf['img'].values == df_noisy_vs_clean['img'].values):
        raise RuntimeError("The input dataframe should be aligned with the ground truth dataframe.")
    tau = scipy.stats.kendalltau(df_noisy_vs_clean.iou, sdf['iou']).statistic
    return tau

### Load all the predictions, from all epochs and all ensemble members
We run in a loop for pretrained or not, and using augmentations or not (for the ablation study).\
We compute the average and std deviation of the predictions over all the seeds, for each epoch.\
Then, we ensemble by averaging the predictions, and compute the metrics for each epoch again, to see how the ensemble performs.

*Note:* Skip this section and go to the next one if stats are already computed.

In [ ]:
pred_root_dir = Path('../data/external/experiments/jakob/')
seeds = [42 + i for i in range(10)]
n_epochs = 30

In [ ]:
for pretrain in [False, True]:
    for aug in [False, True]:
        sdir = f"ensemble_pretrain_{pretrain}_aug_{aug}"
        pred_dir = pred_root_dir / sdir
        print(f"Checking directory: {pred_dir}")
        if not pred_dir.exists():
            raise RuntimeError(f"Directory {pred_dir} does not exist.")

        epoch_start = 0 if pretrain else 1  # 0 means the initial Inria-based checkpoint
        epochs = list(range(epoch_start, n_epochs + 1))

        preds_avg_per_epoch = []
        preds_std_per_epoch = []
        stats_noisy_per_epoch = []
        stats_clean_per_epoch = []

        for epoch in tqdm(epochs, desc="Processing epochs"):
            # We save all the predictions for the current epoch from all seeds
            # image name -> list of predictions
            preds_all = {k: [] for k in fp_clean_labels}
            for seed in seeds:
                pred_dir_crt_seed = pred_dir / f"cv_iter_0_seed_{seed}/version_0/predictions"

                fp_results = pred_dir_crt_seed / f"saved_predictions_{epoch - 1:03d}.npz"
                if not fp_results.exists():
                    raise RuntimeError(f"File {fp_results} does not exist.")

                # Load the predictions
                results = np.load(fp_results)
                preds = results['preds'].squeeze()  # shape: (5000, 256, 256)
                names = results['names']  # shape: (5000,)

                if not len(preds) == len(names) == 5000:
                    raise RuntimeError(
                        f"Mismatch in number of predictions and names. "
                        f"Got {len(preds)} predictions and {len(names)} names instead of 5000."
                    )

                # Save them to the dictionary
                for i, name in enumerate(names):
                    pred = (preds[i].astype(np.float32) / 255.0)  # predictions were saved as uint8 in [0, 255]

                    # Save the raw probabilities (before thresholding) for averaging later
                    preds_all[name].append(pred)

                    # Compute metrics for this single prediction, w.r.t. both noisy and clean labels
                    pred_b = (pred >= 0.5)
                    mask_noisy = np.array(Image.open(fp_noisy_labels[name])) > 0
                    mask_clean = np.array(Image.open(fp_clean_labels[name])) > 0

                    stats_noisy = compute_image_level_metrics(pred_b, mask_noisy)
                    stats_clean = compute_image_level_metrics(pred_b, mask_clean)

                    # Add the seed, image name and epoch to the stats; then save them
                    stats_noisy['epoch'] = epoch
                    stats_noisy['seed'] = seed
                    stats_noisy['img'] = name
                    stats_noisy_per_epoch.append(stats_noisy)
                    stats_clean['epoch'] = epoch
                    stats_clean['seed'] = seed
                    stats_clean['img'] = name
                    stats_clean_per_epoch.append(stats_clean)

            # Compute the ensemble prediction by averaging
            # first take the average, then threshold
            preds_avg = {}
            preds_std = {}
            for name, preds_list in preds_all.items():
                preds_avg[name] = np.mean(preds_list, axis=0) >= 0.5  # threshold at 0.5
                preds_std[name] = np.std(preds_list, axis=0)
            preds_avg_per_epoch.append(preds_avg)
            preds_std_per_epoch.append(preds_std)

            import gc

            gc.collect()  # free memory after each epoch

        stats_df_noisy = pd.DataFrame(stats_noisy_per_epoch)
        stats_df_clean = pd.DataFrame(stats_clean_per_epoch)

        if not len(stats_df_noisy) == len(stats_df_clean) == len(seeds) * len(fp_clean_labels) * len(epochs):
            raise RuntimeError(
                f"Mismatch in number of stats computed. "
                f"Got {len(stats_df_noisy)} instead of {len(seeds) * len(fp_clean_labels) * len(epochs)}."
            )

        _cols = ['img', 'seed', 'epoch']
        stats_df_noisy = stats_df_noisy.sort_values(by=_cols)
        stats_df_clean = stats_df_clean.sort_values(by=_cols)

        # move the 3 columns to the front
        stats_df_noisy = stats_df_noisy[_cols + [c for c in stats_df_noisy.columns if c not in _cols]]
        stats_df_clean = stats_df_clean[_cols + [c for c in stats_df_clean.columns if c not in _cols]]

        # save the stats
        fp_out_noisy = pred_dir / 'stats_noisy.h5'
        stats_df_noisy.to_hdf(fp_out_noisy, key='stats', mode='w')
        print(f"Saved noisy stats to {fp_out_noisy}")

        fp_out_clean = pred_dir / 'stats_clean.h5'
        stats_df_clean.to_hdf(fp_out_clean, key='stats', mode='w')
        print(f"Saved clean stats to {fp_out_clean}")

        if not len(seeds) > 1:
            raise RuntimeError("Ensemble stats requested but only one seed found, cannot compute ensemble.")

        stats_noisy_per_epoch = []
        stats_clean_per_epoch = []

        for i_epoch in tqdm(range(len(preds_avg_per_epoch))):
            preds_avg = preds_avg_per_epoch[i_epoch]
            preds_std = preds_std_per_epoch[i_epoch]

            for img_name in preds_avg:
                pred = preds_avg[img_name]
                mask_noisy = np.array(Image.open(fp_noisy_labels[img_name])) > 0
                mask_clean = np.array(Image.open(fp_clean_labels[img_name])) > 0

                stats_noisy = compute_image_level_metrics(pred, mask_noisy)
                stats_clean = compute_image_level_metrics(pred, mask_clean)

                # Add the average std deviation (over all pixels), then separately for noisy and clean pixels
                stats_clean['avg_std'] = np.sqrt(np.mean(preds_std[img_name] ** 2))
                mask_noise_only = mask_noisy != mask_clean
                stats_clean['avg_std_noise_only'] = np.nan
                if np.sum(mask_noise_only) > 30:  # just to avoid very small masks
                    stats_clean['avg_std_noise_only'] = np.sqrt(np.mean(preds_std[img_name][mask_noise_only] ** 2))
                mask_clean_only = mask_noisy == mask_clean
                stats_clean['avg_std_clean_only'] = np.nan
                if np.sum(mask_clean_only) > 30:  # just to avoid very small
                    stats_clean['avg_std_clean_only'] = np.sqrt(np.mean(preds_std[img_name][mask_clean_only] ** 2))

                # Add the epoch and image name to the stats
                stats_noisy['epoch'] = epochs[i_epoch]
                stats_noisy['img'] = img_name
                stats_clean['epoch'] = epochs[i_epoch]
                stats_clean['img'] = img_name

                stats_noisy_per_epoch.append(stats_noisy)
                stats_clean_per_epoch.append(stats_clean)

        stats_df_noisy = pd.DataFrame(stats_noisy_per_epoch)
        stats_df_clean = pd.DataFrame(stats_clean_per_epoch)
        if not len(stats_df_noisy) == len(stats_df_clean) == len(fp_clean_labels) * len(epochs):
            raise RuntimeError(
                f"Mismatch in number of ensemble stats computed. "
                f"Got {len(stats_df_noisy)} instead of {len(fp_clean_labels) * len(epochs)}."
            )

        _cols = ['img', 'epoch']
        stats_df_noisy = stats_df_noisy.sort_values(by=_cols)
        stats_df_clean = stats_df_clean.sort_values(by=_cols)

        # move the 3 columns to the front
        stats_df_noisy = stats_df_noisy[_cols + [c for c in stats_df_noisy.columns if c not in _cols]]
        stats_df_clean = stats_df_clean[_cols + [c for c in stats_df_clean.columns if c not in _cols]]

        # save the stats
        fp_out_noisy = pred_dir / 'stats_noisy_ensemble.h5'
        stats_df_noisy.to_hdf(fp_out_noisy, key='stats', mode='w')
        print(f"Saved noisy ensemble stats to {fp_out_noisy}")

        fp_out_clean = pred_dir / 'stats_clean_ensemble.h5'
        stats_df_clean.to_hdf(fp_out_clean, key='stats', mode='w')
        print(f"Saved clean ensemble stats to {fp_out_clean}")

### Load the dataframes with all the statistics

In [ ]:
pred_root_dir = Path('../data/external/experiments/jakob/')

def tag_df(df, pretrain, aug, kind):
    df = df.copy()
    df['pretrain'] = 'fine-tune (Inria init)' if pretrain else 'scratch'
    df['aug'] = 'yes' if aug else 'no'
    df['kind'] = kind  # 'seed_models' or 'pred_ensemble'
    return df


members_clean, members_noisy = [], []
ens_clean, ens_noisy = [], []

for pretrain in [False, True]:
    for aug in [False, True]:
        sdir = f"ensemble_pretrain_{pretrain}_aug_{aug}"
        pred_dir = pred_root_dir / sdir
        print(f"Loading stats from directory: {pred_dir}")

        # seed models (per-seed stats)
        members_clean.append(
            tag_df(pd.read_hdf(pred_dir / 'stats_clean.h5', key='stats'), pretrain, aug, 'seed_models'))
        members_noisy.append(
            tag_df(pd.read_hdf(pred_dir / 'stats_noisy.h5', key='stats'), pretrain, aug, 'seed_models'))

        # prediction ensemble (averaging predictions across seeds)
        fp_ce = pred_dir / 'stats_clean_ensemble.h5'
        fp_ne = pred_dir / 'stats_noisy_ensemble.h5'
        ens_clean.append(tag_df(pd.read_hdf(fp_ce, key='stats'), pretrain, aug, 'pred_ensemble'))
        ens_noisy.append(tag_df(pd.read_hdf(fp_ne, key='stats'), pretrain, aug, 'pred_ensemble'))

stats_members_clean = pd.concat(members_clean, ignore_index=True)
stats_members_noisy = pd.concat(members_noisy, ignore_index=True)

stats_ens_clean = pd.concat(ens_clean, ignore_index=True)
stats_ens_noisy = pd.concat(ens_noisy, ignore_index=True)
len(stats_members_clean), len(stats_members_noisy), len(stats_ens_clean), len(stats_ens_noisy)

Compute the IOU average and std accross the 10 models, then of the ensembled predictions

In [ ]:
def seed_mean_std_over_seeds(stats_df):
    # stats_df is per-image rows with columns incl: iou, seed, epoch, pretrain, aug
    per_seed = (
        stats_df
        .groupby(['pretrain', 'aug', 'seed', 'epoch'], as_index=False)
        .mean(numeric_only=True)
    )
    summary = (
        per_seed
        .groupby(['pretrain', 'aug', 'epoch'], as_index=False)
        .agg(mean=('iou', 'mean'), std=('iou', 'std'))
    )
    return summary


iou_members_clean = seed_mean_std_over_seeds(stats_members_clean).assign(labels_version='clean', method='10-seed mean')
iou_members_noisy = seed_mean_std_over_seeds(stats_members_noisy).assign(labels_version='noisy', method='10-seed mean')

iou_ens_clean = (
    stats_ens_clean
    .groupby(['pretrain', 'aug', 'epoch'], as_index=False)
    .mean(numeric_only=True)[['pretrain', 'aug', 'epoch', 'iou']]
    .rename(columns={'iou': 'mean'})
    .assign(std=np.nan, labels_version='clean', method='prediction ensemble')
)
iou_ens_noisy = (
    stats_ens_noisy
    .groupby(['pretrain', 'aug', 'epoch'], as_index=False)
    .mean(numeric_only=True)[['pretrain', 'aug', 'epoch', 'iou']]
    .rename(columns={'iou': 'mean'})
    .assign(std=np.nan, labels_version='noisy', method='prediction ensemble')
)

iou_lines = pd.concat([iou_members_clean, iou_members_noisy, iou_ens_clean, iou_ens_noisy], ignore_index=True)

# ribbons only for the 10-seed mean curves
iou_ribbons = (
    iou_lines[iou_lines.method == '10-seed mean']
    .assign(ymin=lambda d: d['mean'] - d['std'], ymax=lambda d: d['mean'] + d['std'])
)

Compute the Kendall tau correlation between the predicted rankings and the ground truth ranking, for both the ensemble members and the ensemble average.

In [ ]:
# Kendall tau for each seed & epoch, separately per (pretrain, aug)
tau_members = (
    stats_members_noisy
    .groupby(['pretrain', 'aug', 'seed', 'epoch'])
    .apply(compute_kendall_tau, include_groups=False)
    .reset_index(name='tau')
)

tau_members_meanstd = (
    tau_members
    .groupby(['pretrain', 'aug', 'epoch'], as_index=False)
    .agg(mean=('tau', 'mean'), std=('tau', 'std'))
)

tau_members_meanstd['method'] = '10-seed mean'

tau_ens = (
    stats_ens_noisy
    .groupby(['pretrain', 'aug', 'epoch'])
    .apply(compute_kendall_tau, include_groups=False)
    .reset_index(name='mean')
)
tau_ens['std'] = np.nan
tau_ens['method'] = 'avg predictions (ensemble)'

tau_lines = pd.concat([tau_members_meanstd, tau_ens], ignore_index=True)

tau_ribbons = (
    tau_lines[tau_lines.method == '10-seed mean']
    .assign(ymin=lambda d: d['mean'] - d['std'], ymax=lambda d: d['mean'] + d['std'])
)

Prepare two horizontal lines for the plots, corresponding to the initial IoU and tau values of the Inria checkpoint.

In [ ]:
# IoU init baselines (clean + noisy) from fine-tune, epoch 0, aug=yes
idx = (
        (iou_lines.pretrain == 'fine-tune (Inria init)') &
        (iou_lines.epoch == 0) &
        (iou_lines.aug == 'yes') &
        (iou_lines.method == '10-seed mean')
)

hl_iou = (
    iou_lines[idx][['pretrain', 'labels_version', 'mean']]
    .rename(columns={'mean': 'yint'})
    .assign(model_kind='Inria initialization')
)

# tau init baseline from fine-tune, epoch 0, aug=yes
idx = (
        (tau_lines.pretrain == 'fine-tune (Inria init)') &
        (tau_lines.epoch == 0) &
        (tau_lines.aug == 'yes') &
        (tau_lines.method == '10-seed mean')
)

hl_tau = (
    tau_lines[idx][['pretrain', 'mean']]
    .rename(columns={'mean': 'yint'})
    .assign(model_kind='Inria initialization')
)

#### Make a single plot with all the results

In [ ]:
linewidth = 0.5
ptsize = 1.0
base_text_size = 9

method_levels = ['mean ± sd (10 seeds)', 'avg predictions (ensemble)']
method_palette = {k: v for (k, v) in zip(method_levels, ('#0072B2', '#D55E00'))} # Okabe–Ito (colorblind friendly)

# maps for renaming in legend
method_rename = {
    '10-seed mean': 'mean ± sd (10 seeds)',
    'prediction ensemble': 'avg predictions (ensemble)',
    'avg predictions (ensemble)': 'avg predictions (ensemble)',
}
pretrain_rename = {
    'scratch': 'Random init',
    'fine-tune (Inria init)': 'Inria init (fine-tuned)',
}
for _df in (iou_lines, iou_ribbons, tau_lines, tau_ribbons):
    _df['method'] = _df['method'].replace(method_rename)
for _df in (iou_lines, iou_ribbons, hl_iou, tau_lines, tau_ribbons, hl_tau):
    _df['pretrain'] = _df['pretrain'].replace(pretrain_rename)

iou_lines['line_group'] = (
        iou_lines['method'].astype(str) + '|' +
        iou_lines['labels_version'].astype(str) + '|' +
        iou_lines['aug'].astype(str)
)
iou_ribbons['rib_group'] = (
        iou_ribbons['method'].astype(str) + '|' +
        iou_ribbons['labels_version'].astype(str) + '|' +
        iou_ribbons['aug'].astype(str)
)

tau_lines['line_group'] = tau_lines['method'].astype(str) + '|' + tau_lines['aug'].astype(str)
tau_ribbons['rib_group'] = tau_ribbons['method'].astype(str) + '|' + tau_ribbons['aug'].astype(str)

# ensure facet order: scratch left, fine-tune right
pretrain_order = list(pretrain_rename.values())
for _df in (iou_lines, iou_ribbons, hl_iou, tau_lines, tau_ribbons, hl_tau):
    _df['pretrain'] = pd.Categorical(_df['pretrain'], categories=pretrain_order, ordered=True)

aug_order = ['no', 'yes']
for _df in (iou_lines, iou_ribbons, tau_lines, tau_ribbons):
    _df['aug'] = pd.Categorical(_df['aug'], categories=aug_order, ordered=True)

hl_iou_txt = iou_lines[iou_lines.epoch == 0].iloc[:1]
hl_iou_txt['epoch'] = n_epochs
hl_iou_txt['y'] = hl_iou.yint.mean()
hl_iou_txt['label'] = "Inria baseline"
p_iou = (
        p9.ggplot() +
        p9.geom_ribbon(
            p9.aes(x='epoch', ymin='ymin', ymax='ymax', group='rib_group', fill='method'),
            data=iou_ribbons, alpha=0.12, inherit_aes=False, show_legend=False
        ) +
        p9.geom_hline(
            p9.aes(yintercept='yint', linetype='labels_version'),
            data=hl_iou, size=linewidth, show_legend=False
        ) +
        p9.geom_text(
            p9.aes(x="epoch", y="y", label="label"),
            data=hl_iou_txt, va="center", ha="right",
            size=base_text_size * 0.9, inherit_aes=False
        ) +
        p9.geom_line(
            p9.aes('epoch', 'mean', color='method', linetype='labels_version', group='line_group'),
            data=iou_lines, size=linewidth
        ) +
        p9.geom_point(
            p9.aes('epoch', 'mean', color='method', linetype='labels_version', shape='aug', group='line_group'),
            data=iou_lines, size=ptsize
        ) +
        p9.facet_wrap('~ pretrain', ncol=2) +
        p9.labs(x='', y='IoU', tag='a)') +
        p9.scale_color_manual(
            name='Aggregation', values=method_palette, breaks=method_levels, limits=method_levels, labels=method_levels
        ) +
        p9.scale_fill_manual(
            name='Aggregation', values=method_palette, breaks=method_levels, limits=method_levels, labels=method_levels
        ) +
        p9.scale_linetype_discrete(name='Evaluation Labels', breaks=['clean', 'noisy'], labels=['clean', 'noisy']) +
        p9.scale_shape_discrete(name='Augmentation', breaks=['no', 'yes'], labels=['no', 'yes']) +
        p9.guides(
            color=p9.guide_legend(order=1, nrow=1, byrow=True),
            shape=p9.guide_legend(order=2, nrow=1),
            linetype=p9.guide_legend(order=3, nrow=1)
        ) +
        p9.theme_bw(base_size=base_text_size) +
        p9.theme(
            axis_text_x=p9.element_blank(),
            axis_ticks_major_x=p9.element_blank(),
            legend_position='top',
            legend_box='vertical',
            legend_direction='horizontal',
            legend_box_just='left',
            legend_justification='left',
            legend_box_spacing=0.01,
            legend_key_height=4,
            figure_size=(9.2, 3.2),
            dpi=200
        )
)

hl_tau_text = tau_lines[tau_lines.epoch == 0].iloc[:1]
hl_tau_text['epoch'] = n_epochs
hl_tau_text['y'] = hl_tau.yint.mean()
hl_tau_text['label'] = "Inria baseline"

p_tau = (
        p9.ggplot() +
        p9.geom_ribbon(
            p9.aes(x='epoch', ymin='ymin', ymax='ymax', group='rib_group', fill='method'),
            data=tau_ribbons, alpha=0.12, inherit_aes=False, show_legend=False
        ) +
        p9.geom_hline(
            p9.aes(yintercept='yint'),
            data=hl_tau, size=linewidth, show_legend=False
        ) +
        p9.geom_text(
            p9.aes(x="epoch", y="y", label="label"),
            data=hl_tau_text, va="bottom", ha="right",
            size=base_text_size * 0.9, inherit_aes=False
        ) +
        p9.geom_line(
            p9.aes('epoch', 'mean', color='method', group='line_group'),
            data=tau_lines, size=linewidth
        ) +
        p9.geom_point(
            p9.aes('epoch', 'mean', color='method', shape='aug', group='line_group'),
            data=tau_lines, size=ptsize
        ) +
        p9.facet_wrap('~ pretrain', ncol=2) +
        p9.labs(x='Epoch', y="Kendall’s τ", tag='b)') +
        p9.scale_color_manual(
            name='Aggregation', values=method_palette, breaks=method_levels, limits=method_levels, labels=method_levels
        ) +
        p9.scale_fill_manual(
            name='Aggregation', values=method_palette, breaks=method_levels, limits=method_levels, labels=method_levels
        ) +
        p9.scale_shape_discrete(name='Augmentation', breaks=['no', 'yes'], labels=['no', 'yes']) +
        p9.theme_bw(base_size=base_text_size) +
        p9.theme(legend_position='none', figure_size=(9.2, 8), dpi=200)
)

p = p_iou / p_tau
p.save(filename=pred_root_dir / 'iou_and_kendall_tau_over_epochs.pdf', dpi=300)
p


#### Export a ranking and test it on kaggle

In [ ]:
# This should correspond to our final submission
_df = stats_ens_noisy[
    (stats_ens_noisy.pretrain == 'fine-tune (Inria init)') &
    (stats_ens_noisy.aug == 'yes') &
    (stats_ens_noisy.epoch == 30)
].copy()
# _df = stats_df_noisy[(stats_df_noisy.model == 'single') & (stats_df_noisy.epoch == 4)].copy()
_df['noise_level'] = 1 - _df.iou
fp = '../data/external/experiments/jakob/ensemble_pretrain_True_aug_True/ensemble_pretrain_True_aug_True_epoch_30_ranking.csv'
# fp = '../data/external/experiments/jakob/ensemble_wo_aug/single_model_noisy_labels_epoch_4_ranking.csv'
_df = _df.rename(columns={'img': 'image_id'})
_df = _df.sort_values(['noise_level', 'image_id'])
_df['id'] = np.arange(len(_df))
_df = _df[['id', 'image_id', 'iou', 'noise_level']]
_df['image_id'] = _df['image_id'].apply(lambda x: x + '.png')
_df.to_csv(fp, index=False)
print(f"Saved ranking to {fp}")
_df